In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

The syntax of the command is incorrect.
'cp' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
!kaggle datasets download -d urbikn/sroie-datasetv2

'kaggle' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!unzip sroie-datasetv2.zip

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
import cv2
import os

input_folder = "SROIE2019/train/img"
output_folder = "preprocessed"

os.makedirs(output_folder, exist_ok=True)

for img_file in os.listdir(input_folder):

    img_path = os.path.join(input_folder, img_file)

    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    thresh = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    cv2.imwrite(os.path.join(output_folder, img_file), thresh)

print("Preprocessing completed")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'SROIE2019/train/img'

In [ ]:
!ls preprocessed | head

In [ ]:
import os
import json
import pandas as pd

entity_folder = "SROIE2019/train/entities"

data = []

for file in os.listdir(entity_folder):

    if file.endswith(".txt"):

        path = os.path.join(entity_folder, file)

        with open(path) as f:

            content = f.read()

        try:
            data_json = json.loads(content)

            address = data_json.get("address","")

            data.append([file.replace(".txt",".jpg"), address])

        except:
            continue


df_address = pd.DataFrame(data, columns=["image","address"])

df_address.to_csv("address_labels.csv", index=False)

print("Address dataset created")
print(df_address.head())

In [ ]:
import os
import pandas as pd

box_folder = "SROIE2019/train/box"
df = pd.read_csv("address_labels.csv")

yolo_labels = []

for index,row in df.iterrows():

    img_name = row["image"]
    address = str(row["address"]).lower()

    box_file = os.path.join(box_folder, img_name.replace(".jpg",".txt"))

    if not os.path.exists(box_file):
        continue

    with open(box_file) as f:

        for line in f.readlines():

            parts = line.strip().split(",")

            coords = list(map(int,parts[:8]))
            text = ",".join(parts[8:]).lower()

            if text in address or address in text:

                yolo_labels.append([img_name,coords,text])


print("Detected address boxes:",len(yolo_labels))

Detected address boxes: 3227


In [ ]:
import os
import cv2
import pandas as pd

box_folder = "SROIE2019/train/box"
image_folder = "preprocessed"
label_folder = "yolo_labels"

os.makedirs(label_folder, exist_ok=True)

df = pd.read_csv("address_labels.csv")

for index,row in df.iterrows():

    img_name = row["image"]
    address = str(row["address"]).lower()

    img_path = os.path.join(image_folder, img_name)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    box_file = os.path.join(box_folder, img_name.replace(".jpg",".txt"))

    if not os.path.exists(box_file):
        continue

    yolo_lines = []

    with open(box_file) as f:

        for line in f.readlines():

            parts = line.strip().split(",")

            coords = list(map(int,parts[:8]))
            text = ",".join(parts[8:]).lower()

            if text in address or address in text:

                x_coords = coords[0::2]
                y_coords = coords[1::2]

                xmin = min(x_coords)
                xmax = max(x_coords)
                ymin = min(y_coords)
                ymax = max(y_coords)

                x_center = ((xmin+xmax)/2)/w
                y_center = ((ymin+ymax)/2)/h
                width = (xmax-xmin)/w
                height = (ymax-ymin)/h

                yolo_lines.append(f"0 {x_center} {y_center} {width} {height}")

    if len(yolo_lines)>0:

        with open(os.path.join(label_folder,img_name.replace(".jpg",".txt")),"w") as f:

            f.write("\n".join(yolo_lines))

print("YOLO labels created")

YOLO labels created


In [ ]:
!ls yolo_labels | head

In [ ]:
!cat yolo_labels/X51007846395.txt

0 0.46805079621044143 0.27394526795895097 0.31445273130417256 0.01282782212086659
0 0.4634146341463415 0.3026653363740023 0.0657125579520258 0.01097491448118586

In [ ]:
import os

os.makedirs("dataset/images/train", exist_ok=True)
os.makedirs("dataset/images/val", exist_ok=True)
os.makedirs("dataset/labels/train", exist_ok=True)
os.makedirs("dataset/labels/val", exist_ok=True)

print("Folders created")

Folders created


In [ ]:
import shutil
import random

images = os.listdir("preprocessed")
random.shuffle(images)

split = int(len(images)*0.8)

train_images = images[:split]
val_images = images[split:]

for img in train_images:

    shutil.copy("preprocessed/"+img, "dataset/images/train/"+img)

    label = img.replace(".jpg",".txt")

    if os.path.exists("yolo_labels/"+label):
        shutil.copy("yolo_labels/"+label, "dataset/labels/train/"+label)


for img in val_images:

    shutil.copy("preprocessed/"+img, "dataset/images/val/"+img)

    label = img.replace(".jpg",".txt")

    if os.path.exists("yolo_labels/"+label):
        shutil.copy("yolo_labels/"+label, "dataset/labels/val/"+label)


print("Dataset split completed")

Dataset split completed


In [ ]:
!pip install ultralytics

In [ ]:
%%writefile data.yaml

path: dataset
train: images/train
val: images/val

names:
  0: address

Writing data.yaml


In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="data.yaml",
    epochs=60,
    imgsz=960,
    batch=8,
    device=0,
    degrees=2,
    scale=0.3,
    fliplr=0.5,
    mosaic=1.0
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=2, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e18c03bc980>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train2/weights/best.pt")

In [ ]:
results = model.predict(
    source="/content/SROIE2019/train/img/X51007846395.jpg",
    conf=0.25,
    save=True
)


image 1/1 /content/SROIE2019/train/img/X51007846395.jpg: 960x704 3 addresss, 25.7ms
Speed: 6.5ms preprocess, 25.7ms inference, 1.4ms postprocess per image at shape (1, 3, 960, 704)
Results saved to /content/runs/detect/predict2


In [ ]:
import numpy as np

boxes = results[0].boxes.xyxy.cpu().numpy()

# Merge all boxes into one big box
x1 = np.min(boxes[:,0])
y1 = np.min(boxes[:,1])
x2 = np.max(boxes[:,2])
y2 = np.max(boxes[:,3])

merged_box = [x1,y1,x2,y2]

In [ ]:
img = cv2.imread(image_path)

x1,y1,x2,y2 = map(int, merged_box)

crop = img[y1:y2, x1:x2]

cv2.imwrite("/content/final_address.jpg", crop)

True

In [ ]:
!pip install easyocr

In [ ]:
import easyocr
import re

reader = easyocr.Reader(['en'])

result = reader.readtext("/content/final_address.jpg", detail=0)

# combine text
text = " ".join(result)

# remove unwanted characters
text = re.sub(r'[^A-Za-z0-9,./ ]', ' ', text)

# remove extra spaces
text = re.sub(r'\s+', ' ', text).strip()

# fix split words
words = text.split()

fixed_words = []
i = 0

while i < len(words):
    if len(words[i]) <= 2 and i < len(words)-1:
        merged = words[i] + words[i+1]
        fixed_words.append(merged)
        i += 2
    else:
        fixed_words.append(words[i])
        i += 1

clean_address = " ".join(fixed_words)

print("Final Clean Address:")
print(clean_address)

Final Clean Address:
Level 6, Bangunan TH, Damansara Uptown3 No.3, Jalan SS21/39,47400 Petal ing Seangor Jaya


In [ ]:
import cv2
import numpy as np
import easyocr
import re
import pandas as pd
import glob
from ultralytics import YOLO

In [ ]:
model = YOLO("/content/runs/detect/train2/weights/best.pt")

In [ ]:
reader = easyocr.Reader(['en'])

In [ ]:
image_paths = glob.glob("/content/SROIE2019/train/img/*.jpg")[:100]

In [ ]:
data = []

for image_path in image_paths:

    # YOLO prediction
    results = model.predict(source=image_path, conf=0.25)

    boxes = results[0].boxes.xyxy.cpu().numpy()

    if len(boxes) == 0:
        continue

    # Merge boxes
    x1 = np.min(boxes[:,0])
    y1 = np.min(boxes[:,1])
    x2 = np.max(boxes[:,2])
    y2 = np.max(boxes[:,3])

    # Crop address region
    img = cv2.imread(image_path)

    x1,y1,x2,y2 = map(int, [x1,y1,x2,y2])

    crop = img[y1:y2, x1:x2]

    # OCR
    result = reader.readtext(crop, detail=0)

    text = " ".join(result)

    # Cleaning
    text = re.sub(r'[^A-Za-z0-9,./ ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Fix split words
    words = text.split()

    fixed_words = []
    i = 0

    while i < len(words):
        if len(words[i]) <= 2 and i < len(words)-1:
            merged = words[i] + words[i+1]
            fixed_words.append(merged)
            i += 2
        else:
            fixed_words.append(words[i])
            i += 1

    clean_address = " ".join(fixed_words)

    data.append({
        "image": image_path.split("/")[-1],
        "address": clean_address
    })

    print("Processed:", image_path)

In [ ]:
df = pd.DataFrame(data)

df.to_csv("/content/extracted_addresses_100.csv", index=False)

print("CSV file created successfully")

CSV file created successfully


In [ ]:
import pandas as pd

df = pd.read_csv("/content/extracted_addresses_100.csv")

df.head(20)

,image,address
0,X51007231365.jpg,"No, 35, Jalan Kebudayaan 8Taman University,813..."
1,X51005441402.jpg,"BISTRO CAFE 109 SS21/14 ,DAMANSARA UTAMA GST I..."
2,X51008099089.jpg,"No. 2, Jalan Temenggurg 19/9 Seksyen 9Bandar M..."
3,X51005711442.jpg,"No,2 ,Jalan Temenggung 19/9 Seksyen 9Bandar Ma..."
4,X51007846388.jpg,"12, Jalan Taripoi 7/4,Kawasan Perindustrian Ta..."
5,X51006556650.jpg,"Lot J,Jalan Felabur 23/1, 40300 Shah Alam, 035..."
6,X51005757294.jpg,LOt 1851 41851 BJaLAN KPB Kawasan PERINDUSTRIA...
7,X51005301659.jpg,"Na.113G t15G, JALAN SETIA GEMOILANG UIjeG DAND..."
8,X51005719893.jpg,480 KL
9,X51005444045.jpg,"Lot PaT, JALAM ANGSA Tanan BERKELEY 41150 KLAN..."


In [ ]:
import joblib
from pathlib import Path

OUT = Path(r"D:\Python\Infosys\main\backend\app\models")
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump(model, OUT / "text_field_classifier.joblib")
print("Saved text_field_classifier.joblib")

if "vectorizer" in globals():
    joblib.dump(globals()["vectorizer"], OUT / "vectorizer.joblib")
    print("Saved vectorizer.joblib")
else:
    print("Skipping vectorizer save: 'vectorizer' is not defined in this notebook session.")
